# 04 — Gradio demo (CPU)

Interactive demo for the submission: upload an H&E patch -> predicted class + confidence +
full 9-class distribution + Grad-CAM overlay. Runs on **CPU** to preserve the GPU budget.

In [ ]:
%run shared.ipynb
import os
from pathlib import Path
import numpy as np
from PIL import Image
PERSIST_DIR = Path(os.environ.get("PERSIST_DIR", "/workspace/shared/ft004"))

RUN_NAME = "vit_l_stain"
ck = load_checkpoint(PERSIST_DIR / "checkpoints" / RUN_NAME / "best.pth", map_location="cpu")
cfg, classes = ck["cfg"], ck["classes"]
device = torch.device("cpu")
model = build_model(cfg, num_classes=len(classes))
raw_module(model).load_state_dict(ck["model"])
model.eval()
cam = make_gradcam(model, cfg["model"])
eval_aug = build_eval_aug(cfg["img_size"])
print("demo model ready:", cfg["model"])


In [ ]:
def classify(image, use_macenko=False):
    img = np.array(image.convert("RGB"))
    if use_macenko:
        try:
            img = MacenkoNormalizer(target_img=img)(img)  # self-normalize as a light demo
        except Exception as e:
            logger.warning(f"Macenko skipped ({e})")
    x = eval_aug(image=img)["image"].unsqueeze(0)
    with torch.no_grad():
        probs = model(x).softmax(1)[0]
    top = int(probs.argmax())
    label_desc = f"{classes[top]} — {CLASS_DESCRIPTIONS[classes[top]]}"
    dist = {f"{c} ({CLASS_DESCRIPTIONS[c]})": float(probs[i]) for i, c in enumerate(classes)}
    if cam is not None:
        raw = np.array(Image.fromarray(img).resize((cfg["img_size"], cfg["img_size"])))
        heat, _ = cam(x, class_idx=top)
        overlay = Image.fromarray(overlay_heatmap(raw, heat))
    else:
        overlay = image
    return label_desc, dist, overlay


In [ ]:
_ensure("gradio")
import gradio as gr

with gr.Blocks(title="FINETUNING_004 — Stain-Robust Pathology Classifier") as demo:
    gr.Markdown("# FINETUNING_004 — Stain-Robust Pathology Classifier\n"
                "Upload a 224x224 H&E patch -> tissue class + confidence + Grad-CAM. "
                f"Model: **{cfg['model']}** fine-tuned on NCT-CRC-HE-100K (AMD MI300X).")
    with gr.Row():
        with gr.Column():
            inp = gr.Image(type="pil", label="H&E patch")
            mac = gr.Checkbox(label="Macenko normalize upload", value=False)
            btn = gr.Button("Classify", variant="primary")
        with gr.Column():
            out_label = gr.Markdown(label="Prediction")
            out_dist = gr.Label(num_top_classes=9, label="Class probabilities")
            out_cam = gr.Image(type="pil", label="Grad-CAM overlay")
    btn.click(classify, inputs=[inp, mac], outputs=[out_label, out_dist, out_cam])

demo.launch(server_name="0.0.0.0", server_port=7860, share=False)
